# ccdp — damage-seg Path A training (CarDD + HITL filtered, nc=6)

**Path A:** keep the existing nc=6 head and warm-start from the current `damage_seg.pt`.
Adds HITL examples for the **overlapping classes only** — no head re-init, no big LR
warmup, just more data for the same labels.

CarDD's 6 classes (you must keep this exact order):
```
0 dent  1 scratch  2 crack  3 glass-shatter  4 lamp-broken  5 tire-flat
```

HITL ships 8 damage classes; we map each one to a CarDD index (or `None` to drop it).
You fill in the dict in section 4. Anything mapped to `None` is excluded from training.

### Works on both Colab and Kaggle
- **Colab:** mounts Drive, caches the dataset there, checkpoints to Drive.
- **Kaggle:** uses `RCLONE_CONF` Secret + GDrive for the same purpose.
Cell 1 auto-detects the platform.

### Runtime
~30–40 min/epoch on T4 with combined ~5.8k images, batch 16, imgsz 640.
Path A target: 30–40 epochs warm-start → ~1.5–2 h. Free-tier safe.


## 1. Platform autodetect + GPU check

In [ ]:
import os, sys, pathlib
ON_COLAB  = 'google.colab' in sys.modules or os.path.exists('/content')
ON_KAGGLE = os.path.exists('/kaggle')
PLATFORM  = 'colab' if ON_COLAB else 'kaggle' if ON_KAGGLE else 'local'
print('platform:', PLATFORM)

import torch
print('CUDA:', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'enable GPU')

## 2. Mount durable storage

On Colab → mount `MyDrive`. On Kaggle → install rclone from the `RCLONE_CONF` secret
and use `gdrive:` as a remote. Either way we end up with a `DURABLE_ROOT` path
that survives a kernel crash.

In [ ]:
import os, subprocess, pathlib

if PLATFORM == 'colab':
    from google.colab import drive
    drive.mount('/content/drive')
    DURABLE_ROOT = '/content/drive/MyDrive/ccdp'
    DURABLE_IS_FS = True   # we can write straight to it
elif PLATFORM == 'kaggle':
    DURABLE_ROOT = '/kaggle/working/_drive'   # local staging; rclone mirrors to gdrive
    GDRIVE_REMOTE = 'gdrive:ccdp'
    DURABLE_IS_FS = False
    try:
        from kaggle_secrets import UserSecretsClient
        conf = UserSecretsClient().get_secret('RCLONE_CONF')
        subprocess.run('which rclone || (curl -sL https://rclone.org/install.sh | sudo bash)',
                       shell=True, check=False)
        cfg_dir = pathlib.Path.home() / '.config' / 'rclone'
        cfg_dir.mkdir(parents=True, exist_ok=True)
        (cfg_dir / 'rclone.conf').write_text(conf)
        print('[rclone] configured')
    except Exception as e:
        print(f'[rclone] skipping sync — {e!r}')
        GDRIVE_REMOTE = None
else:
    DURABLE_ROOT = str(pathlib.Path.home() / 'ccdp-durable')
    DURABLE_IS_FS = True

os.makedirs(DURABLE_ROOT, exist_ok=True)
CKPT_ROOT_DURABLE = f'{DURABLE_ROOT}/checkpoints/segmenter'
os.makedirs(CKPT_ROOT_DURABLE, exist_ok=True)
print('durable root:', DURABLE_ROOT)

## 3. Clone repo + install

In [ ]:
WORK = '/content' if PLATFORM == 'colab' else '/kaggle/working' if PLATFORM == 'kaggle' else os.path.expanduser('~')
os.chdir(WORK)
!rm -rf car-crash-fix-amount-predictor
!git clone --depth 1 https://github.com/theDocWho/car-crash-fix-amount-predictor.git
os.chdir(f'{WORK}/car-crash-fix-amount-predictor')
!pip -q install -e '.[ml]' ultralytics

# Symlink checkpoints/segmenter -> durable storage so saves persist.
import pathlib
pathlib.Path('checkpoints').mkdir(exist_ok=True)
link = pathlib.Path('checkpoints/segmenter')
if link.exists() or link.is_symlink(): link.unlink()
if DURABLE_IS_FS:
    link.symlink_to(CKPT_ROOT_DURABLE)
else:
    link.mkdir()                   # local; we'll rclone-mirror periodically
    os.makedirs(CKPT_ROOT_DURABLE, exist_ok=True)
print('checkpoints/segmenter ->', os.readlink(link) if link.is_symlink() else '(local + rclone mirror)')

In [ ]:
# Defensive sys.path nudge — on Colab, `pip install -e .` registers the package
# but the active kernel sometimes doesn't pick up the new entry until the next
# cell. Without this, §5's `from ccdp.data import cardd_yolo` raises ImportError
# on a fresh runtime. Re-runs are harmless.
import sys, os
# WORK was set in cell 6 — re-derive defensively in case the cell was re-run.
_work = '/content' if PLATFORM == 'colab' else '/kaggle/working' if PLATFORM == 'kaggle' else os.path.expanduser('~')
_src = os.path.join(_work, 'car-crash-fix-amount-predictor', 'src')
if _src not in sys.path:
    sys.path.append(_src)

import ccdp
print('ccdp package successfully installed!')
print('Location:', ccdp.__file__)


## 4. **EDIT ME** — HITL → CarDD class remap

Fill in this dict before training. HITL class names are in the dataset's README — set
the value to the matching CarDD index `0..5`, or `None` to drop that class from
training. Anything not listed here is also dropped.

CarDD ordering:
```
0 dent  1 scratch  2 crack  3 glass-shatter  4 lamp-broken  5 tire-flat
```

Below is a starter guess based on common HITL ontologies — **verify against the
actual HITL class names before kicking off a long run.**

In [ ]:
# HITL classTitle (EXACT case — they are capitalised in meta.json) -> CarDD index 0..5 or None
#
# Heads-up: the HITL "Car Parts and Car Damages" dataset has its two folders
# inverted vs their names. The folder literally named "Car parts dataset" is
# where the DAMAGE polygons live (Dent, Scratch, Cracked, ...). The folder
# named "Car damages dataset" actually holds car-part polygons (Quarter-panel,
# Front-bumper, ...). §5/§6 read from "Car parts dataset" for Path A.
HITL_TO_CARDD = {
    # HITL damage class (exact, case-sensitive) -> CarDD index 0..5 or None
    'Dent':         0,   # -> dent
    'Scratch':      1,   # -> scratch
    'Cracked':      2,   # -> crack
    # No HITL equivalents for CarDD's: glass-shatter, lamp-broken, tire-flat
    'Missing part': None,
    'Broken part':  None,   # too generic to map cleanly
    'Flaking':      None,
    'Paint chip':   None,
    'Corrosion':    None,
}
mapped = sum(1 for v in HITL_TO_CARDD.values() if v is not None)
print(f'HITL classes kept: {mapped}/{len(HITL_TO_CARDD)} (rest dropped)')

## 5. Fetch CarDD + HITL

CarDD comes from the repo's data prep. HITL pulls from the Kaggle CC0 mirror
(replace the slug if you use a different one).

In [ ]:
import os, json, pathlib, glob, getpass

HITL_SLUG = 'YOUR_USERNAME/hitl-vehicle-damage'   # <-- update with actual Kaggle slug
HITL_DRIVE_CACHE = f'{DURABLE_ROOT}/data/raw/hitl'
os.makedirs(HITL_DRIVE_CACHE, exist_ok=True)

# --- Kaggle creds ------------------------------------------------------
# New flow: KAGGLE_USERNAME + KAGGLE_KEY env vars (token from Kaggle → Settings → API).
# Fallbacks: Drive token file -> legacy kaggle.json -> interactive prompt.
def _resolve_kaggle_creds():
    if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
        return 'env'
    if PLATFORM == 'kaggle':
        # Kaggle kernels are auto-authenticated for their own datasets — nothing to do.
        return 'kaggle-native'
    # Colab: try the durable token file, then legacy json, then prompt.
    token_file = '/content/drive/MyDrive/ccdp/kaggle_token.txt'
    legacy_json = '/content/drive/MyDrive/kaggle/kaggle.json'
    if os.path.exists(token_file):
        for line in pathlib.Path(token_file).read_text().splitlines():
            line = line.strip()
            if not line or line.startswith('#') or '=' not in line: continue
            k, v = line.split('=', 1)
            os.environ[k.strip()] = v.strip().strip('"').strip("'")
        if os.environ.get('KAGGLE_USERNAME') and os.environ.get('KAGGLE_KEY'):
            return f'drive token file ({token_file})'
    if os.path.exists(legacy_json):
        data = json.loads(pathlib.Path(legacy_json).read_text())
        os.environ['KAGGLE_USERNAME'] = data['username']
        os.environ['KAGGLE_KEY'] = data['key']
        return f'legacy kaggle.json ({legacy_json})'
    print('No Kaggle creds found. Paste them from Kaggle → Settings → API.')
    os.environ['KAGGLE_USERNAME'] = input('KAGGLE_USERNAME: ').strip()
    os.environ['KAGGLE_KEY'] = getpass.getpass('KAGGLE_KEY (token): ').strip()
    if input(f'Save to {token_file} for next session? [y/N] ').strip().lower() == 'y':
        pathlib.Path(token_file).parent.mkdir(parents=True, exist_ok=True)
        pathlib.Path(token_file).write_text(
            f"KAGGLE_USERNAME={os.environ['KAGGLE_USERNAME']}\n"
            f"KAGGLE_KEY={os.environ['KAGGLE_KEY']}\n"
        )
        os.chmod(token_file, 0o600)
        print(f'saved → {token_file}')
    return 'interactive'

source = _resolve_kaggle_creds()
print(f'[kaggle] auth via {source} (user={os.environ.get("KAGGLE_USERNAME","<kaggle-native>")})')

# CarDD — uses our packaged builder
from ccdp.data import cardd_yolo
cardd_yaml = cardd_yolo.build()
print('CarDD yaml:', cardd_yaml)

# HITL — download if not cached on Drive
if not glob.glob(f'{HITL_DRIVE_CACHE}/*'):
    !pip -q install kaggle
    !kaggle datasets download -d {HITL_SLUG} -p {HITL_DRIVE_CACHE} --unzip
else:
    print('HITL cache present:', HITL_DRIVE_CACHE)

## 6. Build the combined YOLO dataset (CarDD + remapped HITL)

We *don't* touch CarDD's labels. For HITL we rewrite each `.txt` label so the class id
column is the CarDD index from `HITL_TO_CARDD` — and skip any label line whose original
class maps to `None`. Skipped images with no remaining boxes are dropped from the manifest.

In [ ]:
import json, random, shutil, pathlib, yaml
from pathlib import Path

OUT_ROOT = Path('data/processed/cardd_hitl_pathA')
# Clear any stale Ultralytics label cache from a prior misconfigured run.
for c in (OUT_ROOT / 'labels' / 'train.cache', OUT_ROOT / 'labels' / 'val.cache'):
    if c.exists(): c.unlink()
# Nuke any previously written labels — we may have a mix of bbox and polygon
# files from earlier runs; we want a clean YOLO-seg-only set this time.
for split in ('train', 'val'):
    lbl_dir = OUT_ROOT / 'labels' / split
    if lbl_dir.exists():
        shutil.rmtree(lbl_dir)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    img_dir = OUT_ROOT / 'images' / split
    img_dir.mkdir(parents=True, exist_ok=True)

# Canonical class order (matches §8's data.yaml below).
CLASS_NAMES = ['dent', 'scratch', 'crack', 'glass-shatter', 'lamp-broken', 'tire-flat']
# CarDD COCO category names normalise to underscored lowercase via the loader;
# map both spellings -> our index.
NAME_TO_IDX = {
    'dent': 0, 'scratch': 1, 'crack': 2,
    'glass_shatter': 3, 'glass-shatter': 3,
    'lamp_broken':   4, 'lamp-broken':   4,
    'tire_flat':     5, 'tire-flat':     5,
}

# --- Step A: CarDD images + POLYGON labels straight from COCO JSON -------
# We bypass cardd_yolo.build()'s .txt outputs (which are bbox-detection format)
# and re-derive segmentation polygons here so they match HITL's format.
CARDD_COCO = Path('data/raw/car-damage-detection/CarDD_release/CarDD_COCO')
assert (CARDD_COCO / 'annotations').exists(), f'CarDD raw not unpacked at {CARDD_COCO}'

CARDD_SPLITS = {'train': ('train2017', 'instances_train2017.json'),
                'val':   ('val2017',   'instances_val2017.json')}

n_cardd_img, n_cardd_lbl = {'train': 0, 'val': 0}, {'train': 0, 'val': 0}
for split, (img_subdir, ann_name) in CARDD_SPLITS.items():
    img_src_dir = CARDD_COCO / img_subdir
    ann_path = CARDD_COCO / 'annotations' / ann_name
    coco = json.loads(ann_path.read_text())
    cat_name = {c['id']: c['name'].strip().lower().replace(' ', '_')
                for c in coco['categories']}
    images_by_id = {im['id']: im for im in coco['images']}
    anns_by_img = {}
    for a in coco['annotations']:
        anns_by_img.setdefault(a['image_id'], []).append(a)

    for img in coco['images']:
        W, H = img.get('width') or 0, img.get('height') or 0
        if not (W and H):
            continue
        img_src = img_src_dir / img['file_name']
        if not img_src.exists():
            continue
        # Symlink image
        tgt_img = OUT_ROOT / 'images' / split / img['file_name']
        if not tgt_img.exists():
            tgt_img.symlink_to(img_src.resolve())
        # Build polygon label lines
        lines = []
        for a in anns_by_img.get(img['id'], []):
            cls_name = cat_name.get(a['category_id'])
            cls_idx = NAME_TO_IDX.get(cls_name)
            if cls_idx is None:
                continue
            seg = a.get('segmentation')
            if not seg:
                continue
            # COCO segmentation: list of polygons, each a flat [x1,y1,x2,y2,...]
            # (Ignore RLE — CarDD ships polygons.)
            if isinstance(seg, dict):
                continue
            for poly in seg:
                if not isinstance(poly, list) or len(poly) < 6:
                    continue
                norm = []
                for i in range(0, len(poly) - 1, 2):
                    norm.append(f'{poly[i]/W:.6f}')
                    norm.append(f'{poly[i+1]/H:.6f}')
                lines.append(' '.join([str(cls_idx), *norm]))
        n_cardd_img[split] += 1
        if lines:
            (OUT_ROOT / 'labels' / split / f"{Path(img['file_name']).stem}.txt").write_text(
                '\n'.join(lines) + '\n'
            )
            n_cardd_lbl[split] += 1

print(f'[cardd] linked  train={n_cardd_img["train"]} images, {n_cardd_lbl["train"]} polygon-labels')
print(f'[cardd] linked  val  ={n_cardd_img["val"]} images, {n_cardd_lbl["val"]} polygon-labels')

# --- Step B: HITL Supervisely JSON -> YOLO-seg polygons -----------------
# NB: 'Car parts dataset' folder actually contains DAMAGE labels (folders
# are inverted in the HITL release). Confirmed via meta.json class names.
hitl_root = Path(HITL_DRIVE_CACHE) / 'Car parts dataset'
assert hitl_root.exists(), f'expected damage labels at {hitl_root}'

ann_files = sorted(hitl_root.rglob('ann/*.json'))
print(f'[hitl] found {len(ann_files)} annotation files')

random.seed(42)
kept, dropped_no_class, dropped_no_image = 0, 0, 0
for ann_path in ann_files:
    ann = json.loads(ann_path.read_text())
    H, W = ann['size']['height'], ann['size']['width']
    lines = []
    for obj in ann.get('objects', []):
        cls = HITL_TO_CARDD.get(obj['classTitle'])
        if cls is None:
            continue
        pts = obj.get('points', {}).get('exterior', [])
        if len(pts) < 3:
            continue
        norm = []
        for x, y in pts:
            norm.append(f'{x/W:.6f}')
            norm.append(f'{y/H:.6f}')
        lines.append(' '.join([str(cls), *norm]))
    if not lines:
        dropped_no_class += 1
        continue

    img_name = ann_path.name[:-5]   # strip trailing ".json"
    img_src  = ann_path.parent.parent / 'img' / img_name
    if not img_src.exists():
        dropped_no_image += 1
        continue

    split = 'val' if random.random() < 0.1 else 'train'
    safe_stem = ann_path.stem.replace(' ', '_')
    safe_stem = safe_stem[:-4] if safe_stem.lower().endswith(('.png', '.jpg')) else safe_stem
    out_img = OUT_ROOT / 'images' / split / f'hitl_{safe_stem}{Path(img_name).suffix}'
    out_lbl = OUT_ROOT / 'labels' / split / (out_img.stem + '.txt')
    if not out_img.exists():
        out_img.symlink_to(img_src.resolve())
    out_lbl.write_text('\n'.join(lines) + '\n')
    kept += 1

print(f'[hitl] kept: {kept}  dropped_no_overlap_class: {dropped_no_class}  '
      f'dropped_missing_image: {dropped_no_image}')

# --- Sanity check + format spot check -----------------------------------
for split in ('train', 'val'):
    imgs = list((OUT_ROOT/'images'/split).glob('*'))
    lbls = list((OUT_ROOT/'labels'/split).glob('*.txt'))
    n_nonempty = sum(1 for p in lbls if p.read_text().strip())
    # Detection-style label has exactly 5 tokens per line; seg has ≥7.
    n_detect = 0
    for p in lbls[:200]:   # spot-check first 200
        for line in p.read_text().splitlines():
            if len(line.split()) == 5:
                n_detect += 1
                break
    print(f'[combined] {split}: {len(imgs)} images, {len(lbls)} labels, '
          f'{n_nonempty} non-empty | detection-format files in first 200: {n_detect}')

# Write combined data.yaml (nc=6 — Path A unchanged)
combined_yaml = OUT_ROOT / 'data.yaml'
combined_yaml.write_text(yaml.safe_dump({
    'path':  str(OUT_ROOT.resolve()),
    'train': 'images/train',
    'val':   'images/val',
    'nc':    6,
    'names': CLASS_NAMES,
}))
print('combined data.yaml:', combined_yaml)

## 7. Fetch the existing damage_seg base weights (warm-start source)

In [ ]:
import os
BASE_SEG = 'checkpoints/production/damage_seg.pt'
os.makedirs('checkpoints/production', exist_ok=True)
# The release asset is named yoloseg.pt (legacy naming); we save it locally as
# damage_seg.pt so the rest of the notebook (and downstream code) can keep
# referring to BASE_SEG consistently.
if not os.path.exists(BASE_SEG) or os.path.getsize(BASE_SEG) < 1_000_000:
    !curl -fL --retry 3 -o {BASE_SEG} \
       https://github.com/theDocWho/car-crash-fix-amount-predictor/releases/download/v0.2.0/yoloseg.pt
print('base seg:', os.path.exists(BASE_SEG),
      os.path.getsize(BASE_SEG) if os.path.exists(BASE_SEG) else 0, 'bytes')

## 8. Train (warm-start, auto-resume via Ultralytics)

Ultralytics has a native `resume=True` that reads its `args.yaml` + `last.pt` and
continues. We bypass `ccdp train yolov8` for that and call `YOLO.train` directly so
we can flip resume on cleanly when a prior run dir is present.

Production values (substitute when you've sanity-checked):
- epochs = 40, batch = 16, imgsz = 640, patience = 15


In [ ]:
RUN_TRAINING = False    # flip to True to actually run
RESUME       = False    # flip to True ONLY to resume an interrupted prior run
SMOKE        = False    # flip to True for a tiny sanity-check run (1 epoch, batch 2, imgsz 320)

# Production training settings
EPOCHS    = 40
BATCH     = 16
IMGSZ     = 640
PATIENCE  = 15

# Smoke override
SMOKE_EPOCHS, SMOKE_BATCH, SMOKE_IMGSZ = 1, 2, 320

if RUN_TRAINING:
    import glob, pathlib, yaml
    from ultralytics import YOLO

    DATA_YAML = pathlib.Path('data/processed/cardd_hitl_pathA/data.yaml').resolve()
    assert DATA_YAML.exists(), f'data.yaml not found at {DATA_YAML} — re-run §6'

    eps   = SMOKE_EPOCHS if SMOKE else EPOCHS
    batch = SMOKE_BATCH  if SMOKE else BATCH
    imgsz = SMOKE_IMGSZ  if SMOKE else IMGSZ
    print(f'[config] epochs={eps} batch={batch} imgsz={imgsz} smoke={SMOKE} resume={RESUME}')

    prior_runs = sorted(glob.glob('checkpoints/segmenter/run_*pathA*'))

    if RESUME:
        # Explicit-resume mode: validate the prior run is real AND trained on the
        # same dataset before handing control to Ultralytics. No silent fallback
        # to COCO128-seg defaults this time.
        assert prior_runs, 'RESUME=True but no prior run dir found under checkpoints/segmenter/'
        prior     = pathlib.Path(prior_runs[-1])
        last      = prior / 'ultralytics' / 'weights' / 'last.pt'
        args_yaml = prior / 'ultralytics' / 'args.yaml'
        assert last.exists(), f'RESUME=True but no last.pt at {last}'
        assert args_yaml.exists(), (
            f'RESUME=True but no args.yaml at {args_yaml} — refusing to resume '
            'because Ultralytics would silently fall back to COCO128-seg defaults. '
            'Either flip RESUME=False to start fresh, or restore args.yaml.'
        )
        prior_args = yaml.safe_load(args_yaml.read_text()) or {}
        prior_data = pathlib.Path(prior_args.get('data', '')).resolve()
        if str(prior_data) != str(DATA_YAML):
            raise SystemExit(
                f'RESUME refused: prior run trained on {prior_data}, '
                f'but current DATA_YAML is {DATA_YAML}. Wipe the prior dir and '
                'start fresh, or align the data paths.'
            )
        print(f'[resume] continuing from {last}  (prior data={prior_data})')
        model   = YOLO(str(last))
        results = model.train(resume=True)
    else:
        # Fresh run: ALWAYS pass explicit data/project/name so we never inherit
        # whatever default Ultralytics has bundled. A prior run dir on disk is
        # harmless — we just create a new timestamped one alongside it. Note that
        # we explicitly do NOT look at prior_runs here; that's the footgun fix
        # vs the previous auto-resume version.
        from ccdp.registry import create_run
        run_dir = create_run(variant='segmenter', tag='pathA_cardd_hitl_nc6',
                             notes=f'Path A: warm-start nc=6, smoke={SMOKE}')
        print(f'[fresh] run dir: {run_dir}')
        if prior_runs:
            print(f'[fresh] note: {len(prior_runs)} prior run dir(s) on disk; ignoring '
                  '(set RESUME=True to continue the most recent one)')
        model   = YOLO(BASE_SEG)
        results = model.train(
            data=str(DATA_YAML),
            epochs=eps,
            batch=batch,
            imgsz=imgsz,
            patience=PATIENCE,
            project=str(run_dir.resolve()),
            name='ultralytics',
            exist_ok=True,
            plots=True,           # save results.csv + curves for §3.3 of submission nb
            verbose=False,
        )
        # Bubble best.pt / last.pt to run_dir root (matches our registry layout)
        ult = run_dir / 'ultralytics' / 'weights'
        for fn in ('best.pt', 'last.pt'):
            src = ult / fn
            dst = run_dir / fn
            if src.exists():
                if dst.is_symlink() or dst.exists(): dst.unlink()
                dst.symlink_to(src.relative_to(run_dir))
    print('[done]')
else:
    print('Skipped — flip RUN_TRAINING=True to execute.')

## 9. (Kaggle only) Background mirror to GDrive

On Colab the run dir lives on Drive already. On Kaggle we mirror `checkpoints/segmenter/`
to `gdrive:ccdp/checkpoints/segmenter/` every 60s.

In [ ]:
if PLATFORM == 'kaggle' and 'GDRIVE_REMOTE' in dir() and GDRIVE_REMOTE:
    import threading, time, glob, os, subprocess
    def _sync():
        while True:
            try:
                for d in glob.glob('checkpoints/segmenter/run_*'):
                    dest = f"{GDRIVE_REMOTE}/checkpoints/segmenter/{os.path.basename(d.rstrip('/'))}"
                    subprocess.run(['rclone','copy','--update','--quiet',d,dest], check=False)
            except Exception as e:
                print(f'[sync] {e!r}')
            time.sleep(60)
    threading.Thread(target=_sync, daemon=True).start()
    print('[gdrive] mirror running every 60s')
else:
    print('[gdrive] not applicable (Colab uses Drive directly)')

## 10. Export final weights

Copies the best.pt from the latest Path A run dir onto durable storage so you can
download + upload to the GitHub release.

In [ ]:
import glob, os, shutil
src = sorted(glob.glob('checkpoints/segmenter/run_*pathA*/best.pt'))
if src:
    real = os.path.realpath(src[-1])
    dst  = f'{DURABLE_ROOT}/damage_seg_pathA.pt'
    shutil.copy(real, dst)
    print(f'exported -> {dst}')
else:
    print('no best.pt — run training first')

## 11. Resume-from-crash recap

- **Colab:** reconnect runtime → re-run cells 1–8. Cell 2 remounts Drive, cell 3
  re-symlinks `checkpoints/segmenter`, cell 8 detects the prior `run_*pathA*` dir on
  Drive and calls `YOLO(last.pt).train(resume=True)` — Ultralytics resumes from the saved
  optimizer state at the next epoch.
- **Kaggle:** Save & Run All again. The rclone cell pulls the prior run dir back into
  `/kaggle/working/checkpoints/segmenter/`, then the same resume detection fires.

If you change `HITL_TO_CARDD` between launches, **delete the prior run dir first** —
Ultralytics' resume assumes the same data + nc; remapping classes mid-run silently
poisons the training.

### Push to release
```bash
cp /content/drive/MyDrive/ccdp/damage_seg_pathA.pt .
gh release upload v0.2.0 damage_seg.pt --clobber   # or a Path-A-specific asset name
```
